# Ingest circuits file
 1. Read the file 
 1. Add Metadata Columns 
     - Source File
     - Ingestion Timestamp
 1. Write to bronze delta table  

In [0]:
dbutils.widgets.text("p_batch_id", "")  
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/1.environment_config

In [0]:
%run ../00-common/2.bronze_helpers

In [0]:
source_file = f'{landing_folder_path}/{v_batch_id}/circuits.csv'
table_name = f'{catalog_name}.{bronze_schema}.circuits'

#### Step 1 - Read the CSV file 

In [0]:
# Schema Validation
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId',   StringType()),
    StructField("url",         StringType()),
    StructField("circuitName", StringType()),
    StructField("lat",         DoubleType()),
    StructField("long",        DoubleType()),
    StructField("locality",    StringType()),
    StructField("country",     StringType())
])

In [0]:
circuit_df = (
    spark.read
    .format('csv')
    .schema(circuits_schema) 
    .option('header','True') 
    .option('mode', 'FAILFAST')
    .load(source_file)  
)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
circuit_df_final = add_file_metadata(circuit_df)

#### Step 3 - Write to bronze delta table

In [0]:
write_to_bronze(circuit_df_final, table_name, v_batch_id)

In [0]:
display(spark.table(table_name)) 